# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [ ]:


import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY").strip()

# Check the key without displaying it
print("Key length:", len(api_key))
print("Contains newline:", "\n" in api_key or "\r" in api_key)

os.environ["GOOGLE_API_KEY"] = api_key


In [ ]:
!pip install -q -U google-genai
print("Gemini SDK installed.")

In [ ]:

# @title User input needed.

# Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section.",
        "Documents can be related to any topic - sports, manga, food, music etc.",
        "Have a minimum of 4 sentences with each document.",
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),

    "artificial_intelligence": (
        "Artificial intelligence is a branch of computer science that develops machines capable of performing tasks that normally require human intelligence.",
        "AI is widely used in healthcare, education, finance, transportation and entertainment.",
        "Machine learning allows AI systems to learn patterns from data and improve their performance.",
        "Generative AI can create text, images, audio and computer code."
    ),

    "machine_learning": (
        "Machine learning is a subset of artificial intelligence that enables computers to learn from data.",
        "Supervised learning uses labelled datasets to train predictive models.",
        "Unsupervised learning identifies hidden patterns in data without labelled outputs.",
        "Common machine learning algorithms include linear regression, decision trees and random forests."
    ),

    "education_technology": (
        "Technology has changed the way students access educational resources and learning materials.",
        "Online learning platforms allow students to attend classes and access recorded lectures remotely.",
        "Artificial intelligence can help students understand difficult concepts through personalized explanations.",
        "Educational technology can improve accessibility, flexibility and learning efficiency."
    ),

    "research_papers": (
        "A research paper presents an organized investigation of a particular academic topic.",
        "Most research papers contain an abstract, introduction, literature review, methodology and conclusion.",
        "Researchers use citations and references to acknowledge previously published work.",
        "A good research paper uses reliable evidence, clear arguments and a structured methodology."
    )
}


In [ ]:

# @title Define a tool

def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""

    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():

        if doc_name == "sample_protocol":
            continue

        # Convert document sentences into searchable text
        if isinstance(content, (tuple, list)):
            content = " ".join(content)

        words = set(query_lower.split())
        match_count = sum(
            1 for word in words if word in content.lower()
        )

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    output = []

    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")

    return "\n\n".join(output)


# Test the search tool
print(search_documents("artificial intelligence"))


In [ ]:

# @title Ask Gemini to search for something

import google.generativeai as genai
import time

# Configure Gemini API
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Initialize the Gemini model with the search tool
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    tools=[search_documents],
    system_instruction="""
    You are an intelligent research assistant.

    Your task is to answer user questions using the provided document collection.

    When you need information, use the search_documents tool.

    Follow these rules:
    1. Search the documents before answering.
    2. Base your answers on the information retrieved.
    3. Provide clear and informative explanations.
    4. Mention the source documents when possible.
    5. Never invent information that is not present in the documents.
    6. If the documents do not contain the answer, clearly say so.
    """
)

print("Model ready:", model.model_name)


In [ ]:

# @title ReAct Agent Loop

import time
from google import genai
from google.genai import types

api_key = os.environ["GOOGLE_API_KEY"].strip()
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=30000)
)
MODEL_NAME = "gemini-3.8-flash"

# Define the search tool
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the document collection for information relevant to a query.",
            parameters={
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": "The topic or question to search for."
                    }
                },
                "required": ["query"]
            }
        )
    ]
)

# Configure tool calling
config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
    system_instruction="""
    You are an intelligent research assistant.

    Always search the available documents before answering.
    Use the search_documents tool when information is needed.
    Answer clearly using the retrieved information.
    Never invent information or sources.
    If the documents do not contain the answer, say so.
    """
)

# Retry API requests if rate limited

def send_with_retry(contents, max_retries=4):

    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )

        except Exception as e:
            error = str(e)

            if ("503" in error or "429" in error) and attempt < max_retries - 1:
                wait = 15 * (attempt + 1)
                print(f"Gemini temporarily unavailable. Retrying in {wait} seconds...")
                time.sleep(wait)
            else:
                raise


# ReAct agent loop
def run_agent(question: str, max_steps: int = 8) -> str:

    print(f"Question: {question}\n" + "-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):

        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Preserve the model response
        contents.append(model_content)

        # Check for tool calls
        calls = [
            p.function_call
            for p in model_content.parts
            if p.function_call
        ]

        # If no tool call, return final answer
        if not calls:
            answer = response.text or "No answer returned."
            print(f"\nFinal answer:\n{answer}")
            return answer

        # Execute requested tools
        result_parts = []

        for call in calls:
            args = dict(call.args)

            print(f"Step {step}: Calling {call.name}({args})")

            if call.name == "search_documents":
                result = search_documents(args["query"])
            else:
                result = "Unknown tool requested."

            print(f"Result preview: {str(result)[:150]}...")

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        # Send tool results back to Gemini
        contents.append(
            types.Content(
                role="user",
                parts=result_parts
            )
        )

    return "Stopped: Maximum agent steps reached."


# Test the agent
answer = run_agent(
    "What is artificial intelligence?",
    max_steps=3
)

In [ ]:

# @title Ask follow up questions

# Questions based on the documents for the agent to answer.
questions = [
    "What are the main applications of artificial intelligence?",
    "What is the difference between supervised and unsupervised learning?",
    "How does artificial intelligence help students in education?",
    "What are the main sections of a research paper?",
    "What are some common machine learning algorithms?",
    "Who invented artificial intelligence?"
]

# Run these questions after the Gemini API quota resets.
# for i, q in enumerate(questions):
#     print("\n" + "=" * 60)
#     run_agent(q)


In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)

In [ ]:
assert len(questions) >= 4
assert "Add a question here." not in questions

print("All follow-up question checks passed!")